# 12 — The Recommendation Prototype

**Milestone 4 · Task 13.** An initial prototype that takes campaign
information and returns a predicted outcome with useful recommendations.

The prototype is a single reusable call:

```python
rec = get_recommender("..")
print(rec.report(my_campaign))
```

Everything underneath — the final Random Forest that scores the campaign,
the Logistic Regression advisor that estimates what each change is worth, the
feature pipeline and the what-if rules — lives in `src/recommend.py`. This notebook is the
**front end**: describe a campaign, get a report.

**To try your own campaign, edit the dictionary in §13.1 and re-run.**

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.recommend import get_recommender

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)

rec = get_recommender("..")      # loads the data (rebuilding it if needed) and fits the model
print("Prototype ready.")

Prototype ready.


## 13.1 Describe a campaign and get a report

**Edit this cell.** Only launch-time information is accepted — things a
creator knows before pressing publish.

In [2]:
my_campaign = dict(
    goal=40000,                                   # dollars
    duration=60,                                  # days, 1 to 60
    reward_tiers=[10, 25, 50],                    # the price of each reward tier
    category="Film & Video",
    subcategory="Documentary",
    state="CA",                                   # two-letter code
    # Optional - leave out if unknown:
    # city="Los Angeles",
    # name="My project title",
    # launch_date="2012-03-01",
)

print(rec.report(my_campaign))

CAMPAIGN ASSESSMENT
  Film & Video / Documentary - CA - goal $40,000 - 60 days - 3 reward tiers ($10,$25,$50)

Predicted chance of reaching the goal: 7%   [High risk]   (final model: Random Forest (tuned + calibrated))
  Held-out campaigns scored 2%-12% actually succeeded 4% of the time (n=349).

Where you stand (Documentary, 2,442 campaigns)
  Goal: $40,000 is at the 91st percentile (median $8,000).
  Campaigns with a similar goal (half to double yours): 34% succeeded (n=523).
  Successful campaigns here typically ran 35 days with 8 reward tiers.

What you could change (estimated effect on your chance)
  [goal        ] Lower the goal to $36,000: 7% (+0.7 pts)
  [goal        ] Lower the goal to $30,000: 9% (+1.9 pts)
  [goal        ] Lower the goal to $24,000: 10% (+3.6 pts)  <- recommended
  [goal        ] Lower the goal to $20,000: 12% (+5.2 pts)
  [duration    ] Shorten the campaign to 45 days: 9% (+2.1 pts)  <- recommended
  [duration    ] Shorten the campaign to 35 days: 10% (+3.5

## 13.2 What goes in

| Field | Required | Notes |
|---|---|---|
| `goal` | yes | Funding target in dollars. Must be positive. |
| `duration` | yes | Campaign length in days, from 1 to 60 (Kickstarter's maximum). |
| `reward_tiers` | yes | A list of reward prices, e.g. `[5, 25, 100]`. The *number* of tiers matters as much as the prices. |
| `category` | yes | One of the 13 categories (see §13.3). |
| `subcategory` | yes | Must belong to the chosen category. |
| `state` | yes | Two-letter US state code. |
| `city` | no | If omitted, the typical value from the training data is used (not zero). **Supplying it can shift the chance by several points** - see §13.6. |
| `name` | no | Project title. A weak signal; if omitted, typical values are used. |
| `launch_date` | no | If omitted, the latest period in the training data is used. |

## What deliberately cannot go in

`updates`, `backers`, `pledged`, `comments` and `funded_percentage` do not
exist at launch, and the prototype **refuses them** rather than ignoring
them (§13.5). It cannot use information it is not given a slot for.

## 13.3 Valid categories and subcategories

In [3]:
pd.DataFrame([{"category": c, "subcategories": ", ".join(s)}
              for c, s in rec.taxonomy.items()])

,category,subcategories
0,Art,"Art, Conceptual Art, Digital Art, Illustration, Mixed Media, Painting, Performance Art, Public A..."
1,Comics,Comics
2,Dance,Dance
3,Design,"Crafts, Design, Graphic Design, Product Design"
4,Fashion,Fashion
5,Film & Video,"Animation, Documentary, Film & Video, Narrative Film, Short Film, Webseries"
6,Food,Food
7,Games,"Board & Card Games, Games, Video Games"
8,Music,"Classical Music, Country & Folk, Electronic Music, Hip-Hop, Indie Rock, Jazz, Music, Pop, Rock, ..."
9,Photography,Photography


Several categories have a subcategory with the same name as the category
itself (`Dance / Dance`, `Film & Video / Film & Video`). That is how the
source data records campaigns filed under a category without a more
specific subcategory, and it is a valid choice.

## 13.4 Using the result in code

`rec.report(...)` returns formatted text. `rec.assess(...)` returns the
underlying object, for when the numbers are needed rather than the prose.

In [4]:
a = rec.assess(my_campaign)
print(f"Predicted chance:  {a.probability:.1%}")
print(f"Risk band:         {a.band}")
if a.local_info:
    print(f"Held-out campaigns scored {a.local_info['low']:.0%}-{a.local_info['high']:.0%} "
          f"actually succeeded {a.local_info['rate']:.0%} of the time "
          f"(n={a.local_info['n']:,})")

Predicted chance:  6.6%
Risk band:         High risk
Held-out campaigns scored 2%-12% actually succeeded 4% of the time (n=349)


In [5]:
# Every option the engine considered, with its predicted effect.
a.options.assign(
    probability=lambda d: (d.probability * 100).round(1),
    gain_pp=lambda d: d.gain_pp.round(1),
    funding_floor=lambda d: d.funding_floor.round(0),
)[["lever", "change", "probability", "gain_pp", "funding_floor", "recommended"]]

,lever,change,probability,gain_pp,funding_floor,recommended
0,goal,"Lower the goal to $36,000",7.3,0.7,2625.0,False
1,goal,"Lower the goal to $30,000",8.5,1.9,2562.0,False
2,goal,"Lower the goal to $24,000",10.3,3.6,2462.0,True
3,goal,"Lower the goal to $20,000",11.8,5.2,2362.0,False
4,duration,Shorten the campaign to 45 days,8.7,2.1,3486.0,True
5,duration,Shorten the campaign to 35 days,10.1,3.5,4054.0,False
6,duration,Shorten the campaign to 28 days,11.0,4.4,4415.0,False
7,reward tiers,"Offer 6 reward tiers ($10,$15,$20,$25,$40,$50)",14.6,8.0,5838.0,True
8,reward tiers,"Offer 8 reward tiers ($10,$15,$20,$25,$30,$40,$50,$100)",20.7,14.1,8279.0,False
9,reward tiers,"Offer 10 reward tiers ($10,$15,$20,$25,$30,$40,$50,$60,$75,$100)",26.3,19.7,10536.0,False


In [6]:
# The combined plan, as a campaign description you can feed straight back in.
if a.plan:
    print(f"Plan: {a.probability:.1%} -> {a.plan['probability']:.1%}")
    for k in ("goal", "duration", "reward_tiers"):
        print(f"  {k:13s} {my_campaign[k]!s:28.28} -> {a.plan['campaign'][k]!s:.60}")
    direct = rec.probability(a.plan['campaign'])
    print(f"\nPlan estimate (RF score + advisor's effect): {a.plan['probability']:.1%}")
    print(f"Random Forest re-scoring the planned campaign directly: {direct:.1%}")

Plan: 6.6% -> 25.8%
  goal          40000                        -> 24000.0
  duration      60                           -> 45.0
  reward_tiers  [10, 25, 50]                 -> [10.0, 15.0, 20.0, 25.0, 40.0, 50.0]

Plan estimate (RF score + advisor's effect): 25.8%
Random Forest re-scoring the planned campaign directly: 8.7%


**Why two numbers can differ.** The plan estimate starts from the Random
Forest's score and adds the advisor's estimate of what the changes are worth.
Re-scoring the changed campaign directly with the forest can give a different
figure, especially for campaigns far from typical (here: a $40,000, 60-day
documentary with three tiers, in the top tenth of goals). Forests flatten out
in sparse regions of the data, so they often under-react to changes there.
The data itself (notebook 10 §11.5) shows large-goal campaigns with more
tiers succeeding several times as often as those with few, which is the
direction the advisor follows. How often and how far the two disagree is
measured on 250 real campaigns in notebook 14 §15.7.

## 13.5 What the prototype refuses

Bad input produces a specific, actionable message rather than a wrong answer
or a stack trace.

In [7]:
bad_inputs = {
    "includes updates (not known at launch)": {**my_campaign, "updates": 12},
    "includes backers (not known at launch)": {**my_campaign, "backers": 300},
    "duration over Kickstarter's 60-day limit": {**my_campaign, "duration": 90},
    "negative goal": {**my_campaign, "goal": -500},
    "subcategory not in the category": {**my_campaign, "subcategory": "Jazz"},
    "unknown state code": {**my_campaign, "state": "ZZ"},
    "missing reward tiers": {k: v for k, v in my_campaign.items() if k != "reward_tiers"},
}
for label, c in bad_inputs.items():
    try:
        rec.probability(c)
        print(f"NOT REFUSED  {label}")
    except ValueError as e:
        print(f"refused  {label}\n         -> {str(e)[:150]}\n")

refused  includes updates (not known at launch)
         -> ['updates'] cannot be used: they are only known after a campaign launches. This tool predicts from launch-time information only.

refused  includes backers (not known at launch)
         -> ['backers'] cannot be used: they are only known after a campaign launches. This tool predicts from launch-time information only.

refused  duration over Kickstarter's 60-day limit
         -> duration must be between 1 and 60 days (Kickstarter's current maximum)

refused  negative goal
         -> goal must be positive

refused  subcategory not in the category
         -> 'Jazz' is not a subcategory of 'Film & Video'. Choose from: ['Animation', 'Documentary', 'Film & Video', 'Narrative Film', 'Short Film', 'Webseries']

refused  unknown state code
         -> Unknown state 'ZZ'. Use a two-letter code.

refused  missing reward tiers
         -> Missing required fields: ['reward_tiers']



## 13.6 How much do the optional fields matter?

City, title and launch date are optional because a creator may not want to
supply them. But "optional" is not the same as "irrelevant", so measure it:
for every held-out campaign, compare the prediction with the real value
against the prediction with the field left out.

In [8]:
import numpy as np
from src.recommend import RAW_COLUMNS
from src.prepare import load_splits

_, test = load_splits("..")["random"]
raw = test[RAW_COLUMNS].copy()
with_all = rec.predict_proba(raw)

def effect_of_omitting(label, **blank):
    r = raw.copy()
    for k, v in blank.items():
        r[k] = v
    d = np.abs(rec.predict_proba(r) - with_all) * 100
    return {"field left out": label, "mean change (pts)": d.mean(),
            "90th percentile (pts)": np.percentile(d, 90), "largest (pts)": d.max()}

omit = pd.DataFrame([
    effect_of_omitting("city", city=None),
    effect_of_omitting("title", name=None),
    effect_of_omitting("launch date", launch_date=rec.default_launch_date),
    effect_of_omitting("all three", city=None, name=None,
                       launch_date=rec.default_launch_date),
])
omit.round(1)

,field left out,mean change (pts),90th percentile (pts),largest (pts)
0,city,3.9,9.5,28.7
1,title,1.3,3.0,10.6
2,launch date,1.1,2.6,14.9
3,all three,4.5,10.3,40.3


**City is not negligible.** Leaving it out shifts a typical prediction by
several points and some by more than ten. The title and launch date matter
much less. So: **supply the city when it is known** — the probability is
noticeably better informed with it.

Does it also change the *advice*? Every what-if holds city, title and date
fixed, so the recommended changes should mostly survive. Test it on real
campaigns, with and without those three fields:

In [9]:
from src.recommend import campaign_from_row

sample = test[test["duration_days"] <= 60].sample(200, random_state=7)
same = 0
plan_diff = []
for _, row in sample.iterrows():
    full_c = campaign_from_row(row)
    bare_c = {k: v for k, v in full_c.items() if k not in ("city", "name", "launch_date")}
    a_full, a_bare = rec.assess(full_c), rec.assess(bare_c)
    r_full = set(a_full.options.loc[a_full.options["recommended"], "change"])
    r_bare = set(a_bare.options.loc[a_bare.options["recommended"], "change"])
    same += r_full == r_bare
    plan_diff.append(abs((a_full.plan["gain_pp"] if a_full.plan else 0.0)
                         - (a_bare.plan["gain_pp"] if a_bare.plan else 0.0)))

print(f"Campaigns tested:                         {len(sample)}")
print(f"Identical recommended changes:            {same / len(sample):.0%}")
print(f"Plan gain differs by (mean / 90th / max): "
      f"{np.mean(plan_diff):.1f} / {np.percentile(plan_diff, 90):.1f} / {np.max(plan_diff):.1f} points")

Campaigns tested:                         200
Identical recommended changes:            90%
Plan gain differs by (mean / 90th / max): 1.0 / 2.5 / 8.4 points


The advice is **largely but not completely** insensitive to the optional
fields. Most campaigns get the same recommendations either way; the starting
probability is what moves. A borderline option can cross the 2-point
threshold when the baseline shifts, because the same change in odds is worth
slightly different points at different starting probabilities.

(September's notebooks called location a modest signal. That was wrong, for
the state and the city alike: see notebook 10 section 11.6.)

## 13.7 Scoring several campaigns at once

`summary_frame` takes a dictionary of named campaigns and returns one row
each — useful for comparing alternatives before committing to one.

In [10]:
options = {
    "As planned": my_campaign,
    "Smaller goal": {**my_campaign, "goal": 25000},
    "Shorter": {**my_campaign, "duration": 35},
    "More reward tiers": {**my_campaign, "reward_tiers": [10, 15, 25, 40, 50, 100, 250]},
    "All three": {**my_campaign, "goal": 25000, "duration": 35,
                  "reward_tiers": [10, 15, 25, 40, 50, 100, 250]},
}
summary = rec.summary_frame(options)
summary.assign(chance=(summary.chance * 100).round(1),
               **{"plan chance": (summary["plan chance"] * 100).round(1),
                  "plan gain (pts)": summary["plan gain (pts)"].round(1)})

,campaign,chance,band,levers advised,plan chance,plan gain (pts)
0,As planned,6.6,High risk,"goal, duration, reward tiers",25.8,19.2
1,Smaller goal,7.0,High risk,"goal, duration, reward tiers",26.7,19.7
2,Shorter,7.0,High risk,"goal, reward tiers",22.1,15.1
3,More reward tiers,9.4,High risk,"goal, duration, reward tiers",29.2,19.8
4,All three,13.8,High risk,"goal, duration, reward tiers",30.1,16.2


## 13.8 What this prototype is and is not

**It is** a way to put a campaign next to 38,000 others from 2009-2012, see
where it sits, and see which changes the data associates with better
outcomes.

**It is not**

- **A guarantee or a cause.** Campaigns that did these things succeeded more
  often, partly because the creators behind them were also better prepared
  in ways the data cannot see. Every report says so.
- **A view of today's Kickstarter.** The model learned from US campaigns
  launched between 2009 and 2012. Launch year is clipped to that range so
  the model is never asked to extrapolate.
- **Aware of the project itself.** It sees a goal, a duration and some
  prices. It does not see the video, the description, the creator's
  following or their track record — which plausibly matter more than
  anything here, and which this dataset does not contain.

## Task 13 — Summary

- One call — `rec.report(campaign)` — turns a campaign description into a
  probability from the final Random Forest, a risk band with the observed rate for that range, a position
  among comparable campaigns, ranked what-if options, and a combined plan.
- The same engine is available as an object (`rec.assess`) and as a batch
  comparison (`rec.summary_frame`).
- Bad or leaky input is refused with a specific message.
- Optional inputs are optional but not irrelevant: the city shifts the
  starting probability by several points, so supply it when known. The
  *advice* is unaffected because every what-if holds them fixed.

**Next:** `13_test_and_improve.ipynb` tests the prototype on sample and real
campaigns and records what testing changed.